# Speaker verification on AfriVoices-KE: download, split and voiceprints, calibration and the one-time test (up to Gate 4)

Runs `sv_01_collect_afrivoices.py`, `sv_02_process.py` to `sv_06_figures.py` from `SPEAKER_VERIFICATION_PLAN.md`. The model is frozen, so nothing is trained.
**Stop after the last cell.** Gates 3a and 3 passed, and the rule is frozen in `config.py`. The test speakers can be scored **once only**.

Before running:
1. Upload `sv_code.zip` to `MyDrive/mzazi_sv/` in Google Drive.
2. Add a Colab secret (key icon, left bar) named `HF_TOKEN` holding a Hugging Face read token, and allow this notebook to access it.
3. A CPU runtime is enough.

The audio is AfriVoices-KE (Anv-ke), CC BY 4.0. Downloads, recordings and results are written to `MyDrive/mzazi_sv/`, so if the session drops, running all cells again picks up from the finished blocks. Keep the tab open: Colab stops idle sessions.

In [ ]:
from google.colab import drive, userdata
drive.mount("/content/drive")

import os, pathlib, shutil, zipfile
DRIVE = pathlib.Path("/content/drive/MyDrive/mzazi_sv")
CODE = pathlib.Path("/content/acoustic-model")
shutil.rmtree(CODE, ignore_errors=True)
zipfile.ZipFile(DRIVE / "sv_code.zip").extractall(CODE)

# Data and this module's results live on Drive; the code stays on the local disk.
(DRIVE / "data").mkdir(exist_ok=True)
(DRIVE / "results").mkdir(exist_ok=True)
(CODE / "data").symlink_to(DRIVE / "data")
(CODE / "results" / "speaker_verification").symlink_to(DRIVE / "results")

os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")  # read by huggingface_hub, never printed
os.environ["PYTHONPATH"] = "src:scripts"
os.environ["SV_DATASET"] = "afrivoices"
os.chdir(CODE)

In [ ]:
!pip install -q -r requirements-speaker.txt
# CPU builds of PyTorch, as in the local venv-speaker: the default Linux builds need CUDA, which a CPU
# runtime lacks. The +cpu versions are named exactly and reinstalled, so a CUDA copy is always replaced.
!pip install -q --force-reinstall --no-deps torch==2.14.1+cpu torchaudio==2.11.1+cpu --index-url https://download.pytorch.org/whl/cpu
!python -c "import torch, torchaudio; print(torch.__version__, torchaudio.__version__, 'CUDA:', torch.version.cuda)"

In [ ]:
!python -m pytest -q tests/speaker_verification

## Download

About 93 GB is streamed and about 1,300 recordings are kept. Progress goes to `MyDrive/mzazi_sv/data/speaker_verification/afrivoices/collect.log`.

In [ ]:
!python scripts/sv_01_collect_afrivoices.py

In [ ]:
print(open("results/speaker_verification/afrivoices/01_data.md", encoding="utf-8").read())

## Split (Gate 3a)

In [ ]:
!python scripts/sv_02_process.py

In [ ]:
for name in sorted(pathlib.Path("results/speaker_verification/afrivoices").glob("*.md")):
    print(f"===== {name.name}\n" + name.read_text(encoding="utf-8"))

Gate 3a passed on 2026-11-12. After a disconnect, run cells 1 and 2 and then carry on from here: the download and split are kept on Drive.

## Voiceprints, calibration speakers only (Gate 3)

Embeds every calibration speaker with the frozen ECAPA-TDNN and scores the calibration visits. Test speakers are not touched. Scores are cached on Drive, so a rerun after a disconnect resumes.

In [ ]:
!python scripts/sv_03_embed.py

In [ ]:
print(open("results/speaker_verification/afrivoices/03_embeddings.md", encoding="utf-8").read())

Gate 3 passed on 2026-11-13: E1 is confirmed.

## Calibration (SV4)

Chooses the threshold on calibration visits only, from the scores above. Takes seconds.

In [ ]:
!python scripts/sv_04_calibrate.py

SV4 rule frozen on 2026-11-13: cosine 0.5, at least 5 s of speech per recording.

## The one-time test (SV5)

First, the test speakers' voiceprints and visits are scored under all five conditions. This is the slow step, cached on Drive, so a rerun after a disconnect resumes. It prints only timings.

In [ ]:
!python scripts/sv_03_embed.py

Then the evaluation. **This runs once:** it claims the test set, and a second run is refused. If it fails before finishing, the test set is not used and it can be run again.

In [ ]:
!python scripts/sv_05_evaluate.py

In [ ]:
!python scripts/sv_06_figures.py

**Stop here.** Share the output of the evaluation cell for Gate 4. The charts are in `MyDrive/mzazi_sv/results/afrivoices/figures/`.